# Peat forest disturbances and their drivers in the Republic of Congo and DRC, 2022 onward
*Method adapted from Nesha et al. (2024), Environ. Res. Lett. 19 104031, [doi:10.1088/1748-9326/ad6679](https://doi.org/10.1088/1748-9326/ad6679), with wall-to-wall WUR driver maps in place of visual interpretation.*

**Questions**
1. Where and when do peat forest disturbances (PFD) occur? → 1 km intensity grid, annual maps, hotspot districts, monthly patterns and temporal hotspots
2. How do protection status and accessibility relate to them? → protected areas, concessions, distance to peat edges, roads and rivers
3. What drives them, and how do drivers vary in space and time? → WUR driver classes by country, year, month, event size, protection and proximity

**Data**
| Component | Source |
|---|---|
| Peat | Gumbricht et al. (2017) ∪ Crezee et al. (2022); agreement / non-agreement kept |
| Forest | RADD 2018 primary humid forest baseline (Turubanova et al. 2018) |
| Disturbances | `DISTURBANCE_SOURCE = "radd"`: high-confidence RADD alerts. `"wur"`: WUR integrated alerts (RADD + GLAD-L + GLAD-S2, + DIST from 2026) |
| Drivers | WUR: `distribution/driverclassification_afr_202201_202512` (2022–2025) + `operational_postprocessed_afr_collection` (2026 →), 11 classes |
| Protection | National PAs (Forest Atlas, if uploaded), WDPA, WRI managed forest concessions (if uploaded) |
| Access | Roads: Kleinschroth et al. (2019) if uploaded, else GRIP4; rivers: GLCF inland water |
| Peat forest type | Crezee et al. (2022) palm / hardwood (optional) |

Area, intensity and seasonality are computed per pixel. Protection and proximity are computed per disturbance event (8-connected pixels), with pixel-area versions alongside.

**Workflow.** Part A builds the layers in Earth Engine and launches three batch exports. Part B loads the results and produces maps (§13) and graphs and tables (§14). All figures are saved as PNG and all tables as CSV in `outputs_peat_disturbance/`.

## 0 · Setup

In [ ]:
import json, math, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ee

EE_PROJECT = "fao-sepal-361210"   # processing project (SEPAL); assets live in ee-cocoacmr
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine API", ee.__version__, "| project:", EE_PROJECT)

## 1 · Configuration
Everything you are likely to change is in this cell. Paths marked **UPLOAD** are not in the public EE catalog. Each layer falls back gracefully if missing, and the notebook tells you which analyses are affected.

In [ ]:
# ── Output locations ───────────────────────────────────────────────────────
ASSET_ROOT = "projects/ee-cocoacmr/assets/peat_disturbance"
LOCAL_OUT = Path("outputs_peat_disturbance"); LOCAL_OUT.mkdir(exist_ok=True)
OVERWRITE = False

# ── Period & alert confidence ──────────────────────────────────────────────
START_YEAR, END_YEAR = 2022, 2026       # WUR drivers start Jan 2022; 2026 is partial (operational layer, monthly)
ALERT_CONFIDENCE = [3]                  # RADD mode: high confidence only (3); [2, 3] adds low-confidence alerts
DISTURBANCE_SOURCE = "radd"             # "radd" (WUR drivers attached where dates match) | "wur" (WUR integrated alerts, 2022 →)

# ── Area of interest ───────────────────────────────────────────────────────
AOI_MODE = "countries"                  # "countries" (full national territories) | "cuvette_centrale" | "custom"
COUNTRY_LIST = ["Congo", "Democratic Republic of the Congo"]     # FAO GAUL ADM0_NAME (ROC, DRC)
CUVETTE_AOI_ASSET = None                # UPLOAD (optional): Cuvette Centrale outline
CUVETTE_BBOX = [14.5, -4.0, 21.5, 3.5]  # fallback approximation of that outline
CUSTOM_AOI_ASSET = None

# ── Peat maps, forest mask, peat forest type ───────────────────────────────
PEAT_MASK_MODE = "union"                # "union" (combined map) | "agreement" | "none" (all humid forest)
PEAT_GUMBRICHT = {"asset": "projects/ee-cocoacmr/assets/peat/gumbricht2017_peat",   # UPLOAD, 231 m
                  "band": None, "peat_values": None}       # None = any value > 0 is peat
PEAT_CREZEE = {"asset": "projects/ee-cocoacmr/assets/peat/crezee2022_peat",         # UPLOAD, 50 m
               "band": None, "min_value": 0}               # value > min_value is peat
PEAT_VEG = {"asset": "projects/ee-cocoacmr/assets/peat/crezee2022_vegetation",      # UPLOAD (optional)
            "band": None, "palm_values": [], "hardwood_values": []}   # class codes of the Crezee land-cover map
EDGE_REFERENCE = "peat"                 # distance from the combined peat map edge (Nesha et al.); or "peat_forest"

# ── Drivers (WUR) ──────────────────────────────────────────────────────────
DRIVERS_HIST_ASSET = "projects/wurnrt-drivers/assets/distribution/driverclassification_afr_202201_202512"  # 2022–2025
DRIVERS_OPER_ASSET = "projects/wurnrt-drivers/assets/operational_postprocessed_afr_collection"          # 2026 →, monthly
DRIVER_BAND, DRIVER_DATE_BAND = "class", "date"          # date = YYdoy
DRIVER_START_YEAR = 2022
DRIVER_CLASSES = {
    1: "Small-scale agriculture", 2: "Small-scale agriculture (fire)",
    3: "Large-scale agriculture", 4: "Large-scale agriculture (fire)",
    5: "Road development", 6: "Selective logging", 7: "Mining", 8: "Flooding",
    9: "Wildfire", 10: "Other natural disturbance", 11: "Uncertain classification"}
FIRE_SUBTYPES = {2: 1, 4: 3}            # 'with fire' subtype → parent class
DRIVER_DATE_TOLERANCE_DAYS = 120        # RADD mode: max gap between RADD date and WUR alert date to accept the label
DRIVER_GROUP_COLORS = {                 # fire subtypes are folded into their parent class for maps and graphs
    "Small-scale agriculture": "#e6ab02", "Large-scale agriculture": "#a6761d", "Road development": "#525252",
    "Selective logging": "#1b9e77", "Mining": "#7570b3", "Flooding": "#1f78b4", "Wildfire": "#d95f02",
    "Other natural disturbance": "#66a61e", "Uncertain classification": "#bdbdbd"}

# ── Protection status ──────────────────────────────────────────────────────
WDPA = "WCMC/WDPA/current/polygons"     # world protected areas
NATIONAL_PA_ASSET = None                # UPLOAD: Forest Atlas national PAs (MECNDD & MEDD-DIAF 2023); None = split WDPA
CONCESSIONS_ASSET = None                # UPLOAD: WRI managed forest concessions (2021); None skips that analysis

# ── Accessibility ──────────────────────────────────────────────────────────
ROADS_ASSET = "projects/ee-cocoacmr/assets/access/kleinschroth2019_congo_roads"    # UPLOAD (Kleinschroth et al. 2019)
ROADS_FALLBACK = "projects/sat-io/open-datasets/GRIP4/Africa"                      # community catalog — verify path
RIVERS_MODE = "glcf"                    # "glcf" (GLCF inland water, Feng et al. 2016) | "hydrosheds"
GLCF_WATER = "GLCF/GLS_WATER"           # band "water": 2 = water
HYDRORIVERS = "WWF/HydroSHEDS/v1/FreeFlowingRivers"; RIVER_MAX_ORDER = 6

# ── Analysis parameters ────────────────────────────────────────────────────
ADMIN0, ADMIN2 = "FAO/GAUL/2015/level0", "FAO/GAUL/2015/level2"
EDGE_BINS_KM = [0.5, 1, 2, 3, 4, 5, 10]                  # km
ACCESS_BINS_KM = [1, 2, 3, 4, 5, 10, 15, 20, 25]         # km
EDGE_SCALE, ACCESS_SCALE = 50, 100                       # m
EDGE_NEIGH_PX, ACCESS_NEIGH_PX = 256, 300                # search radius in pixels (12.8 km, 30 km)
SIZE_CLASSES_HA = [0.5, 2]                               # small < 0.5 ≤ medium ≤ 2 < large
INTENSITY_THRESHOLDS_PCT = [2, 20]                       # low < 2 % ≤ medium ≤ 20 % < high (spatial hotspot)
TEMPORAL_HOTSPOT_PCT = 10                                # month with > 10 % of the year's PFD

# ── Maps ───────────────────────────────────────────────────────────────────
MAP_RES_DEG = 0.01                      # finest display resolution (~1 km); coarsened automatically for large AOIs
SHOW_PA_ON_MAPS = True
DOM_CELL_DEG = 0.1                      # cell size for the dominant-driver map
MIN_CELL_HA = 1                         # hide dominant-driver cells with less disturbance than this
MAX_POINTS = 100_000                    # max events plotted per driver panel
EVENTS_SPLIT_BY_YEAR = False                             # False = connected pixels over the whole period
MMU_HA = 0.1                                             # RADD MMU (0.2 ha before Jul 2020); smaller events are intersection artefacts (App. L)

# ── Compute ────────────────────────────────────────────────────────────────
TILE_DEG, TILE_SCALE = 1.0, 8
STATS_SCALE, PF_SCALE = 10, 30
TEST_BOX = [18.4, 2.8, 18.9, 3.3]       # dry-run box (NW Sud-Ubangi) — move if empty

# ── Derived ────────────────────────────────────────────────────────────────
COUNTRIES = COUNTRY_LIST
if START_YEAR < DRIVER_START_YEAR:
    print(f"ℹ WUR drivers start in {DRIVER_START_YEAR} — START_YEAR moved from {START_YEAR}")
    START_YEAR = DRIVER_START_YEAR
YEARS = list(range(START_YEAR, END_YEAR + 1)); N_YEARS = len(YEARS)
CONF_SIZE = 2 if (DISTURBANCE_SOURCE == "radd" and len(set(ALERT_CONFIDENCE)) > 1) else 1
CONF_TAG = "c" + "".join(map(str, sorted(ALERT_CONFIDENCE))) if DISTURBANCE_SOURCE == "radd" else "wur"
RUN_TAG = f"{AOI_MODE}_{PEAT_MASK_MODE}_{START_YEAR}_{END_YEAR}_{CONF_TAG}"
print("Run tag:", RUN_TAG, "| countries:", COUNTRIES)

## 2 · Helpers

In [ ]:
def asset_type(path):
    """'IMAGE', 'IMAGE_COLLECTION', 'TABLE', 'FOLDER' or None (missing / no access)."""
    if not path:
        return None
    try:
        return ee.data.getAsset(path)["type"]
    except ee.EEException:
        return None

def ensure_folder(path):
    if asset_type(path) is None:
        ee.data.createAsset({"type": "FOLDER"}, path); print("created folder", path)

def load_image(asset, band=None):
    t = asset_type(asset)
    if t not in ("IMAGE", "IMAGE_COLLECTION"):
        return None
    img = ee.ImageCollection(asset).mosaic() if t == "IMAGE_COLLECTION" else ee.Image(asset)
    return img.select(band) if band else img.select(0)

def doy_to_month(year_img, doy_img):
    leap = year_img.mod(4).eq(0)
    doy = doy_img.subtract(leap.And(doy_img.gt(59)))
    month = ee.Image(1)
    for cum in [31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]:
        month = month.add(doy.gt(cum))
    return month.int()

def yydoy_to_days(img):
    """YYdoy → approximate day count, for date differences."""
    return img.divide(1000).floor().multiply(365.25).floor().add(img.mod(1000))

def bin_index(dist_img, edges):
    b = ee.Image(0)
    for e in edges:
        b = b.add(dist_img.gt(e))
    return b.int()

def cum_labels(edges):
    return [f"≤{e:g} km" for e in edges] + ["all"]

def encode(imgs, fields):
    """Mixed-radix encode categorical bands into one int32 code band."""
    code = ee.Image(0).int()
    for name, size in fields:
        code = code.multiply(size).add(ee.Image(imgs[name]).int())
    return code.int()

def decode(codes, fields):
    rem = pd.Series(codes).round().astype("int64").to_numpy().copy()
    out = {}
    for name, size in reversed(fields):
        out[name] = rem % size
        rem //= size
    return pd.DataFrame(out)[[n for n, _ in fields]]

def distance_km(source, scale, neigh_px):
    """Euclidean distance (km) to the nearest non-zero pixel of `source`, at `scale` m.
    At these ranges it differs from Haversine distance by well under 1 %."""
    far = neigh_px * scale / 1000.0
    d = (source.unmask(0).gt(0)
         .fastDistanceTransform(neigh_px, "pixels", "squared_euclidean")
         .sqrt().multiply(ee.Image.pixelArea().sqrt()).divide(1000)
         .reproject(crs="EPSG:4326", scale=scale))
    return d.unmask(far)

TASKS = {}
def start_export(name, make_task, asset_id):
    if asset_type(asset_id) is not None:
        if not OVERWRITE:
            print(f"[skip] {asset_id} exists (set OVERWRITE=True to rebuild)"); return None
        ee.data.deleteAsset(asset_id)
    task = make_task(); task.start(); TASKS[name] = task
    print(f"[started] {name}: {task.id}"); return task

def task_status():
    return pd.DataFrame([{"export": n, "state": t.status().get("state"),
                          "error": t.status().get("error_message", "")} for n, t in TASKS.items()])

def fc_to_df(fc):
    return ee.data.computeFeatures({"expression": ee.FeatureCollection(fc), "fileFormat": "PANDAS_DATAFRAME"})

def pct(s):
    return s / s.sum() * 100

def cum_share(df, col, edges, weight="area_ha"):
    """Cumulative % by distance bin; weight=None counts rows (events)."""
    n = len(edges) + 1
    s = (df.groupby(col)[weight].sum() if weight else df.groupby(col).size()).reindex(range(n), fill_value=0)
    return pd.Series((s.cumsum() / s.sum() * 100).to_numpy(), index=cum_labels(edges))

## 3 · WUR driver maps — load and check
This section combines the historical (2022–2025) and operational (2026 onward) maps into one collection, keeping the `class` and `date` bands. It then checks access, bands, time coverage and the class codes present against the documented legend (1–11).

In [ ]:
def load_driver_collection(asset):
    t = asset_type(asset)
    if t == "IMAGE_COLLECTION": return ee.ImageCollection(asset), t
    if t == "IMAGE": return ee.ImageCollection([ee.Image(asset)]), t
    return None, t

dcol = None
for name, asset in [("historical", DRIVERS_HIST_ASSET), ("operational", DRIVERS_OPER_ASSET)]:
    col, t = load_driver_collection(asset)
    print(f"{name}: {asset} → {t}")
    if col is None:
        print("   ⚠ not readable — check that your account and the SEPAL project have read access"); continue
    first = col.first()
    print("   images:", col.size().getInfo(), "| bands:", first.bandNames().getInfo(), "| types:", first.bandTypes().getInfo())
    ts = [x for x in col.aggregate_array("system:time_start").getInfo() if x is not None]
    if ts:
        print("   time_start:", pd.to_datetime(min(ts), unit="ms").date(), "→", pd.to_datetime(max(ts), unit="ms").date())
    col = col.map(lambda i: i.select([DRIVER_BAND, DRIVER_DATE_BAND]).toInt32())
    dcol = col if dcol is None else dcol.merge(col)
if dcol is None:
    raise RuntimeError("Neither driver asset is readable")

probe = dcol.map(lambda i: i.updateMask(i.select(DRIVER_BAND).gt(0))).mosaic()
reg = ee.Geometry.Rectangle([8, -6, 31, 8])
hist = probe.select(DRIVER_BAND).reduceRegion(reducer=ee.Reducer.frequencyHistogram(), geometry=reg, scale=1000,
                                             maxPixels=1e10, bestEffort=True).getInfo()[DRIVER_BAND] or {}
found_codes = sorted(int(float(k)) for k in hist)
print("Class codes found (1 km probe):", found_codes)
unexpected = set(found_codes) - set(DRIVER_CLASSES)
if unexpected:
    print("⚠ Codes not in DRIVER_CLASSES (will count as no_driver_alert):", unexpected)
mm = probe.select(DRIVER_DATE_BAND).reduceRegion(reducer=ee.Reducer.minMax(), geometry=reg, scale=1000,
                                                maxPixels=1e10, bestEffort=True).getInfo()
print("Alert date range (YYdoy):", mm)

# Part A · Build layers in Earth Engine
## 4 · Area of interest, countries, tiles

In [ ]:
gaul0 = ee.FeatureCollection(ADMIN0)
found = gaul0.filter(ee.Filter.inList("ADM0_NAME", COUNTRIES)).aggregate_array("ADM0_NAME").distinct().getInfo()
assert set(found) == set(COUNTRIES), f"Not found in GAUL: {set(COUNTRIES) - set(found)}"

countries_fc = ee.FeatureCollection([
    ee.Feature(gaul0.filter(ee.Filter.eq("ADM0_NAME", n)).geometry(100), {"cidx": i, "name": n})
    for i, n in enumerate(COUNTRIES)])

if AOI_MODE == "cuvette_centrale":
    aoi = (ee.FeatureCollection(CUVETTE_AOI_ASSET).geometry(100) if asset_type(CUVETTE_AOI_ASSET) == "TABLE"
           else ee.Geometry.Rectangle(CUVETTE_BBOX))
    if asset_type(CUVETTE_AOI_ASSET) != "TABLE":
        print("⚠ Using the approximate Cuvette bounding box")
elif AOI_MODE == "custom":
    aoi = ee.FeatureCollection(CUSTOM_AOI_ASSET).geometry(100)
else:
    aoi = countries_fc.geometry(100)
aoi_bounds = aoi.bounds(100)

country_img = ee.Image().int().paint(countries_fc, "cidx").rename("cidx")
if AOI_MODE in ("cuvette_centrale", "custom"):
    country_img = country_img.clip(aoi)

ring = aoi_bounds.coordinates().getInfo()[0]
xmin, xmax = math.floor(min(p[0] for p in ring)), math.ceil(max(p[0] for p in ring))
ymin, ymax = math.floor(min(p[1] for p in ring)), math.ceil(max(p[1] for p in ring))
tiles, tid, x = [], 0, xmin
while x < xmax:
    y = ymin
    while y < ymax:
        tiles.append(ee.Feature(ee.Geometry.Rectangle([x, y, min(x + TILE_DEG, xmax), min(y + TILE_DEG, ymax)],
                                                      None, False), {"tid": tid}))
        tid += 1; y += TILE_DEG
    x += TILE_DEG
tiles_fc = ee.FeatureCollection(tiles).filterBounds(aoi)
print("Tiles intersecting AOI:", tiles_fc.size().getInfo())

gaul2 = ee.FeatureCollection(ADMIN2).filter(ee.Filter.inList("ADM0_NAME", COUNTRIES)).filterBounds(aoi)

## 5 · RADD, forest baseline, combined / agreement peat forest, peat forest type
The peat maps are used at native resolution, and EE samples them nearest-neighbour at the 10 m scale of the reductions, which is equivalent to resampling them to 10 m. `peat_src` records 1 = Gumbricht only, 2 = Crezee only and 3 = both; 1 and 2 together form the *non-agreement* class.

In [ ]:
radd = ee.ImageCollection("projects/radar-wur/raddalert/v1")
radd_alert = (radd.filterMetadata("layer", "contains", "alert")
              .filterMetadata("geography", "contains", "africa")
              .sort("system:time_end", False).first())
radd_baseline = (radd.filterMetadata("layer", "contains", "forest_baseline")
                 .filterMetadata("geography", "contains", "africa").first())
RADD_PROJ = radd_alert.select("Alert").projection()
LATEST_ALERT_DATE = ee.Date(radd_alert.get("system:time_end")).format("YYYY-MM-dd").getInfo()
print("RADD alert image:", radd_alert.get("system:index").getInfo(), "| valid to", LATEST_ALERT_DATE)
if int(LATEST_ALERT_DATE[:4]) <= END_YEAR:
    print(f"⚠ END_YEAR {END_YEAR} is incomplete in RADD — recent months lack confirmed high-confidence alerts")

def peat_layer(cfg, kind):
    img = load_image(cfg.get("asset"), cfg.get("band"))
    if img is None:
        print(f"⚠ {kind} peat map not found: {cfg.get('asset')}")
        return ee.Image(0).byte(), False
    if kind == "Gumbricht" and cfg.get("peat_values"):
        vals = cfg["peat_values"]; m = img.remap(vals, [1] * len(vals), 0)
    elif kind == "Gumbricht":
        m = img.gt(0)
    else:
        m = img.gt(cfg.get("min_value", 0))
    return m.unmask(0).byte(), True

g_peat, HAS_G = peat_layer(PEAT_GUMBRICHT, "Gumbricht")
c_peat, HAS_C = peat_layer(PEAT_CREZEE, "Crezee")
peat_src = g_peat.add(c_peat.multiply(2)).rename("peat_src")

if PEAT_MASK_MODE == "union":
    if not (HAS_G or HAS_C): raise RuntimeError("No peat map available for PEAT_MASK_MODE='union'")
    peat_mask = peat_src.gt(0)
elif PEAT_MASK_MODE == "agreement":
    if not (HAS_G and HAS_C): raise RuntimeError("Both peat maps are needed for PEAT_MASK_MODE='agreement'")
    peat_mask = peat_src.eq(3)
else:
    peat_mask = ee.Image(1).byte()

forest2018 = radd_baseline.select(0).gt(0).unmask(0)
pf_nocountry = forest2018.And(peat_mask)
peat_forest = pf_nocountry.updateMask(country_img.mask()).selfMask().rename("peat_forest")

# peat forest type (Crezee): 0 other / unknown · 1 palm-dominated · 2 hardwood-dominated
veg_raw = load_image(PEAT_VEG.get("asset"), PEAT_VEG.get("band"))
HAS_VEG = veg_raw is not None and PEAT_VEG["palm_values"] and PEAT_VEG["hardwood_values"]
if HAS_VEG:
    veg_img = (veg_raw.remap(PEAT_VEG["palm_values"], [1] * len(PEAT_VEG["palm_values"]), 0)
               .add(veg_raw.remap(PEAT_VEG["hardwood_values"], [2] * len(PEAT_VEG["hardwood_values"]), 0))
               .min(2).unmask(0).byte().rename("veg"))
else:
    veg_img = ee.Image(0).byte().rename("veg")
    print("ℹ No peat vegetation-type layer — palm/hardwood breakdown (App. K) skipped")

## 6 · WUR driver layer
Monthly images are combined per pixel with `qualityMosaic` on `date`, so the **most recent** WUR alert and its class are kept. Driver index values:
* 0: no WUR alert at the pixel
* 1–11: WUR classes
* `date_mismatch` (RADD mode): a WUR alert exists, but its date is more than `DRIVER_DATE_TOLERANCE_DAYS` from the RADD date

In [ ]:
drv_latest = dcol.map(lambda i: i.updateMask(i.select(DRIVER_BAND).gt(0))).qualityMosaic(DRIVER_DATE_BAND)
drv_class, drv_date = drv_latest.select(DRIVER_BAND), drv_latest.select(DRIVER_DATE_BAND)

CLASS_CODES = sorted(DRIVER_CLASSES)
DRIVER_LABELS = ["no_driver_alert"] + [DRIVER_CLASSES[c] for c in CLASS_CODES] + ["date_mismatch"]
ND = len(DRIVER_LABELS)
IDX_MISMATCH = ND - 1
NON_DRIVER = {"no_driver_alert", "date_mismatch"}
drv_idx_raw = drv_class.remap(CLASS_CODES, list(range(1, len(CLASS_CODES) + 1)), 0)
print("Driver index → label:", dict(enumerate(DRIVER_LABELS)))

## 7 · Peat forest disturbances (PFD)
* **`radd`**: RADD alerts with confidence in `ALERT_CONFIDENCE`, within the year window, inside peat forest. A WUR label is attached when the WUR alert date is within tolerance of the RADD date.
* **`wur`**: WUR integrated alert pixels inside peat forest, dated by the WUR `date` band. Every pixel has a label. This captures disturbances that GLAD-L, GLAD-S2 or DIST detect but RADD misses.

In [ ]:
if DISTURBANCE_SOURCE == "radd":
    alert_conf = radd_alert.select("Alert")
    alert_date = radd_alert.select("Date")
    conf_ok = alert_conf.remap(ALERT_CONFIDENCE, [1] * len(ALERT_CONFIDENCE), 0)
else:
    alert_conf = ee.Image(3)
    alert_date = drv_date
    conf_ok = drv_class.gt(0)

alert_year = alert_date.divide(1000).floor().add(2000).int().rename("year")
alert_doy = alert_date.mod(1000).int()
alert_month = doy_to_month(alert_year, alert_doy).rename("month")
dist_mask = (conf_ok.And(alert_year.gte(START_YEAR)).And(alert_year.lte(END_YEAR))
             .And(peat_forest.unmask(0))).selfMask().rename("dist")

if DISTURBANCE_SOURCE == "radd":
    has_drv = drv_class.gt(0).unmask(0)
    gap = yydoy_to_days(alert_date).subtract(yydoy_to_days(drv_date)).abs()
    matched = has_drv.And(gap.lte(DRIVER_DATE_TOLERANCE_DAYS).unmask(0))
    driver_idx = ee.Image(0).where(has_drv, IDX_MISMATCH).where(matched, drv_idx_raw.unmask(0))
else:
    driver_idx = drv_idx_raw.unmask(0)
driver_idx = driver_idx.int().rename("driver")

## 8 · Protection status and accessibility
Following Nesha et al. (2024):
* **Peat edge distance** is measured from the edge of the combined peat map (`EDGE_REFERENCE="peat"`), so edge pixels are 0 and values grow towards the interior.
* **Roads** use Kleinschroth et al. (2019) if uploaded, otherwise GRIP4. **Rivers** use the GLCF inland water dataset. Road, river and combined distances are kept separately to reproduce Fig. 5.
* **Protected areas:** *national* come from the Forest Atlas if uploaded (otherwise the non-international WDPA designations), and *world* come from WDPA. Codes: 0 none · 1 national · 2 world · 3 both.

In [ ]:
# peat edge
if EDGE_REFERENCE == "peat" and PEAT_MASK_MODE != "none":
    edge_inside = peat_mask
else:
    edge_inside = pf_nocountry
edge_km = distance_km(edge_inside.Not(), EDGE_SCALE, EDGE_NEIGH_PX).rename("edge_km")

# roads
ROADS_USED = next((a for a in (ROADS_ASSET, ROADS_FALLBACK) if asset_type(a) == "TABLE"), None)
print("Roads:", ROADS_USED or "⚠ none readable — road distance will be 'far' everywhere")
road_src = (ee.Image(0).byte().paint(ee.FeatureCollection(ROADS_USED).filterBounds(aoi_bounds), 1)
            if ROADS_USED else ee.Image(0).byte())
road_km = distance_km(road_src, ACCESS_SCALE, ACCESS_NEIGH_PX).rename("road_km")

# rivers / inland water
if RIVERS_MODE == "glcf":
    water30 = (ee.ImageCollection(GLCF_WATER).filterBounds(aoi_bounds).mosaic().select("water").eq(2)
               .unmask(0).setDefaultProjection(crs="EPSG:4326", scale=30))
    river_src = (water30.reduceResolution(ee.Reducer.max(), maxPixels=64)
                 .reproject(crs="EPSG:4326", scale=ACCESS_SCALE))      # keep narrow channels at 100 m
else:
    rivers = (ee.FeatureCollection(HYDRORIVERS).filterBounds(aoi_bounds)
              .filter(ee.Filter.lte("RIV_ORD", RIVER_MAX_ORDER)))
    river_src = ee.Image(0).byte().paint(rivers, 1)
river_km = distance_km(river_src, ACCESS_SCALE, ACCESS_NEIGH_PX).rename("river_km")
access_km = road_km.min(river_km).rename("access_km")

# protected areas
wdpa = (ee.FeatureCollection(WDPA).filterBounds(aoi_bounds)
        .filter(ee.Filter.neq("STATUS", "Proposed")).filter(ee.Filter.neq("MARINE", "2")))
if asset_type(NATIONAL_PA_ASSET) == "TABLE":
    pa_nat, pa_world = ee.FeatureCollection(NATIONAL_PA_ASSET).filterBounds(aoi_bounds), wdpa
    print("PAs: national = Forest Atlas, world = WDPA")
else:
    pa_nat = wdpa.filter(ee.Filter.neq("DESIG_TYPE", "International"))
    pa_world = wdpa.filter(ee.Filter.eq("DESIG_TYPE", "International"))
    print("ℹ No Forest Atlas layer — national = WDPA national/regional designations, world = WDPA international (Ramsar, WHS, MAB)")
pa_img = (ee.Image(0).byte().paint(pa_nat, 1).add(ee.Image(0).byte().paint(pa_world, 2))).rename("pa")

HAS_CONC = asset_type(CONCESSIONS_ASSET) == "TABLE"
conc_img = (ee.Image(0).byte().paint(ee.FeatureCollection(CONCESSIONS_ASSET).filterBounds(aoi_bounds), 1)
            if HAS_CONC else ee.Image(0).byte()).rename("conc")
if not HAS_CONC:
    print("⚠ No concession layer — concession analysis skipped")

## 9 · Encode categories into code bands
Every disturbed pixel gets an integer code that combines all of its categories. A grouped `sum` then returns the area for every combination, and Part B only needs pandas `groupby`. Road and river bins go into a second code so the code space stays within int32.

In [ ]:
NBE, NBA = len(EDGE_BINS_KM) + 1, len(ACCESS_BINS_KM) + 1
edge_bin = bin_index(edge_km, EDGE_BINS_KM).rename("edge_bin")
access_bin = bin_index(access_km, ACCESS_BINS_KM).rename("access_bin")
road_bin = bin_index(road_km, ACCESS_BINS_KM).rename("road_bin")
river_bin = bin_index(river_km, ACCESS_BINS_KM).rename("river_bin")
conf_band = alert_conf.eq(3) if CONF_SIZE == 2 else ee.Image(0)

FIELDS = [("cidx", len(COUNTRIES)), ("year", N_YEARS), ("month", 12), ("driver", ND), ("conf", CONF_SIZE),
          ("pa", 4), ("conc", 2), ("edge_bin", NBE), ("access_bin", NBA), ("peat_src", 4), ("veg", 3)]
ACC_FIELDS = [("cidx", len(COUNTRIES)), ("year", N_YEARS), ("driver", ND), ("road_bin", NBA), ("river_bin", NBA)]
PF_FIELDS = [("cidx", len(COUNTRIES)), ("peat_src", 4), ("veg", 3)]
for nm, f in [("main", FIELDS), ("access", ACC_FIELDS)]:
    cap = int(np.prod([s for _, s in f]))
    assert cap < 2**31, f"{nm} code space {cap:,} exceeds int32 — reduce bins or years"
    print(f"{nm} code space: {cap:,}")

BANDS = {"cidx": country_img, "year": alert_year.subtract(START_YEAR), "month": alert_month.subtract(1),
         "driver": driver_idx, "conf": conf_band, "pa": pa_img, "conc": conc_img, "edge_bin": edge_bin,
         "access_bin": access_bin, "road_bin": road_bin, "river_bin": river_bin, "peat_src": peat_src, "veg": veg_img}
area_ha = ee.Image.pixelArea().divide(1e4).rename("area")
dist_stack = area_ha.addBands(encode(BANDS, FIELDS).rename("code")).updateMask(dist_mask)
acc_stack = area_ha.addBands(encode(BANDS, ACC_FIELDS).rename("code")).updateMask(dist_mask)
pf_stack = area_ha.addBands(encode(BANDS, PF_FIELDS).rename("code")).updateMask(peat_forest)

LABELS = {
    "cidx": dict(enumerate(COUNTRIES)),
    "year": {i: y for i, y in enumerate(YEARS)},
    "month": {i: i + 1 for i in range(12)},
    "driver": dict(enumerate(DRIVER_LABELS)),
    "conf": ({0: "low", 1: "high"} if CONF_SIZE == 2 else
             {0: "n/a (WUR)" if DISTURBANCE_SOURCE == "wur" else ("high" if ALERT_CONFIDENCE == [3] else "low")}),
    "pa": {0: "none", 1: "national", 2: "world", 3: "national+world"},
    "conc": {0: "outside", 1: "inside"},
    "peat_src": {0: "not peat", 1: "Gumbricht only", 2: "Crezee only", 3: "both maps"},
    "veg": {0: "other/unknown", 1: "palm-dominated", 2: "hardwood-dominated"},
}
def label(df):
    out = df.copy()
    for c, m in LABELS.items():
        if c in out:
            out[c] = pd.Series(out[c]).round().astype(int).map(m)
    if "peat_src" in out:
        out["agreement"] = np.where(out["peat_src"] == "both maps", "agreement",
                           np.where(out["peat_src"] == "not peat", "not peat", "non-agreement"))
    return out.rename(columns={"cidx": "country"})

### 9b · Dry run on a small box (seconds)
This catches asset and permission problems before the large exports.

In [ ]:
tb = ee.Geometry.Rectangle(TEST_BOX)
r = dist_stack.reduceRegion(ee.Reducer.sum().group(1, "code"), tb, scale=30, maxPixels=1e9).getInfo()
g = pd.DataFrame(r.get("groups", []))
if g.empty:
    print("No disturbances in TEST_BOX — move it to a known hotspot")
else:
    t = label(decode(g["code"], FIELDS)); t["area_ha"] = g["sum"].values
    print(f"{t.area_ha.sum():.1f} ha disturbed in test box (30 m approximation)")
    display(t.groupby(["year", "driver"]).area_ha.sum().unstack(fill_value=0).round(1))
    display(t.groupby(["agreement", "pa"]).area_ha.sum().unstack(fill_value=0).round(1))

### 9c · Optional map preview (needs `geemap`)

In [ ]:
try:
    import geemap
    m = geemap.Map(center=[0.5, 18.5], zoom=6)
    m.addLayer(peat_src.selfMask(), {"min": 1, "max": 3, "palette": ["66bd63", "a6d96a", "1a9850"]},
               "Peat (1 Gumbricht, 2 Crezee, 3 both)", False)
    m.addLayer(peat_forest, {"palette": ["1b7837"]}, "Peat forest 2018")
    m.addLayer(dist_mask, {"palette": ["e31a1c"]}, "PFD")
    m.addLayer(driver_idx.updateMask(dist_mask), {"min": 0, "max": ND - 1, "palette": ["bdbdbd", "ffd92f", "e5c494", "e41a1c", "fb9a99",
               "a65628", "1b9e77", "984ea3", "377eb8", "ff7f00", "4daf4a", "666666", "000000"]}, "Driver index")
    m.addLayer(edge_km.updateMask(peat_forest), {"min": 0, "max": 5, "palette": ["fff7bc", "fe9929", "662506"]},
               "Distance to peat edge (km)", False)
    m.addLayer(pa_world.style(color="ffff00", fillColor="00000000"), {}, "World PAs")
    m.addLayer(pa_nat.style(color="ff4500", fillColor="00000000"), {}, "National PAs")
    display(m)
except ImportError:
    print("geemap not installed — skipping preview")

## 10 · Batch exports
1. **stats**: grouped areas per tile. `kind = dist` holds PFD by all categories, `kind = acc` holds PFD by road and river bin, and `kind = pf` holds the peat-forest denominators.
2. **events**: 8-connected PFD pixels. Each event is exported as a centroid with its area, first alert date, and the modal value of every category. Events that cross a tile boundary are split; with 1° tiles this affects few of them.
3. **grid**: a 1 km image with peat forest ha, PFD ha for all years, and PFD ha per year.

In [ ]:
ensure_folder(ASSET_ROOT)
STATS_ASSET = f"{ASSET_ROOT}/stats_{RUN_TAG}"
EVENTS_ASSET = f"{ASSET_ROOT}/events_{RUN_TAG}"
GRID_ASSET = f"{ASSET_ROOT}/grid1km_{RUN_TAG}"

# 1 · grouped statistics
def reduce_tile(tile):
    geom, tid = tile.geometry(), tile.get("tid")
    grp = ee.Reducer.sum().group(1, "code")
    to_fc = lambda groups, kind: ee.FeatureCollection(ee.List(groups).map(
        lambda d: ee.Feature(None, ee.Dictionary(d)).set({"kind": kind, "tid": tid})))
    rd = dist_stack.reduceRegion(grp, geom, scale=STATS_SCALE, maxPixels=1e13, tileScale=TILE_SCALE)
    ra = acc_stack.reduceRegion(grp, geom, scale=STATS_SCALE, maxPixels=1e13, tileScale=TILE_SCALE)
    rp = pf_stack.reduceRegion(grp, geom, scale=PF_SCALE, maxPixels=1e13, tileScale=TILE_SCALE)
    return to_fc(rd.get("groups"), "dist").merge(to_fc(ra.get("groups"), "acc")).merge(to_fc(rp.get("groups"), "pf"))

stats_fc = tiles_fc.map(reduce_tile).flatten()
start_export("stats", lambda: ee.batch.Export.table.toAsset(
    collection=stats_fc, description=f"stats_{RUN_TAG}", assetId=STATS_ASSET), STATS_ASSET)

# 2 · events
EV_CATS = ["driver", "cidx", "pa", "conc", "edge_bin", "access_bin", "road_bin", "river_bin", "peat_src", "veg"]
seg = alert_year if EVENTS_SPLIT_BY_YEAR else ee.Image(1).int()
ev_img = (seg.rename("seg").addBands(alert_date.rename("t_first"))
          .addBands([ee.Image(BANDS[k]).int().rename(k) for k in EV_CATS]).updateMask(dist_mask))
ev_reducer = ee.Reducer.min().setOutputs(["t_first"])
for k in EV_CATS:
    ev_reducer = ev_reducer.combine(ee.Reducer.mode().setOutputs([k]), "", False)

def tile_events(tile):
    v = ev_img.reduceToVectors(reducer=ev_reducer, geometry=tile.geometry(), scale=10,
                               geometryType="polygon", eightConnected=True, labelProperty="seg",
                               maxPixels=1e11, tileScale=TILE_SCALE)
    props = ["seg", "t_first"] + EV_CATS
    return v.map(lambda f: ee.Feature(f.geometry().centroid(1), f.toDictionary(props))
                 .set({"area_ha": f.geometry().area(1).divide(1e4), "tid": tile.get("tid")}))

events_fc = tiles_fc.map(tile_events).flatten()
start_export("events", lambda: ee.batch.Export.table.toAsset(
    collection=events_fc, description=f"events_{RUN_TAG}", assetId=EVENTS_ASSET), EVENTS_ASSET)

# 3 · 1 km intensity grid (10 m → 100 m → 1 km means; ratio of PFD pixels to peat forest pixels, §2.2)
def frac_1km(mask):
    m = mask.unmask(0).float().setDefaultProjection(RADD_PROJ)
    m = m.reduceResolution(ee.Reducer.mean(), maxPixels=1024).reproject(crs="EPSG:4326", scale=100)
    return m.reduceResolution(ee.Reducer.mean(), maxPixels=1024).reproject(crs="EPSG:4326", scale=1000)

cell_ha = ee.Image.pixelArea().divide(1e4)
grid_img = (frac_1km(peat_forest).multiply(cell_ha).rename("pf_ha")
            .addBands(frac_1km(dist_mask).multiply(cell_ha).rename("dist_ha_all"))
            .addBands([frac_1km(dist_mask.And(alert_year.eq(y))).multiply(cell_ha).rename(f"dist_ha_{y}")
                       for y in YEARS]).toFloat())
start_export("grid", lambda: ee.batch.Export.image.toAsset(
    image=grid_img, description=f"grid1km_{RUN_TAG}", assetId=GRID_ASSET, region=aoi_bounds,
    scale=1000, crs="EPSG:4326", maxPixels=1e13, pyramidingPolicy={".default": "mean"}), GRID_ASSET)

## 11 · Monitor exports
Re-run until all three are `COMPLETED`. Runs over the full extent of both countries can take a few hours.

In [ ]:
display(task_status() if TASKS else "No tasks started in this session — outputs may already exist")
for a in (STATS_ASSET, EVENTS_ASSET, GRID_ASSET):
    print(("✓ " if asset_type(a) else "… ") + a)

# Part B · Analysis
## 12 · Load results

In [ ]:
stats_raw = fc_to_df(STATS_ASSET)
def unpack(kind, fields):
    s = stats_raw[stats_raw["kind"] == kind]
    df = label(decode(s["code"], fields)); df["area_ha"] = s["sum"].to_numpy()
    return df.groupby([c for c in df.columns if c != "area_ha"], as_index=False).area_ha.sum()

D, ACC, PF = unpack("dist", FIELDS), unpack("acc", ACC_FIELDS), unpack("pf", PF_FIELDS)
for name, df in [("pfd_stats", D), ("pfd_road_river", ACC), ("peat_forest_area", PF)]:
    df.to_csv(LOCAL_OUT / f"{name}_{RUN_TAG}.csv", index=False)

EV = fc_to_df(EVENTS_ASSET)
for c in ["driver", "cidx", "pa", "conc", "peat_src", "veg", "edge_bin", "access_bin", "road_bin", "river_bin"]:
    EV[c] = EV[c].round().astype(int)
EV = label(EV)
EV["year"] = (EV["t_first"] // 1000 + 2000).astype(int)
SIZE_LABELS = [f"small (<{SIZE_CLASSES_HA[0]:g} ha)", f"medium ({SIZE_CLASSES_HA[0]:g}–{SIZE_CLASSES_HA[1]:g} ha)",
               f"large (≥{SIZE_CLASSES_HA[1]:g} ha)"]
EV["size_class"] = pd.cut(EV.area_ha, [0] + SIZE_CLASSES_HA + [np.inf], right=False, labels=SIZE_LABELS)
EV["below_mmu"] = EV.area_ha < MMU_HA
EV["lon"] = EV["geo"].apply(lambda g: g["coordinates"][0]); EV["lat"] = EV["geo"].apply(lambda g: g["coordinates"][1])
EV.drop(columns="geo").to_csv(LOCAL_OUT / f"pfd_events_{RUN_TAG}.csv", index=False)

# driver groups: fire subtypes folded into their parent class, with a separate fire flag
FIRE_LABELS = [DRIVER_CLASSES[c] for c in FIRE_SUBTYPES]
PARENT = {DRIVER_CLASSES[k]: DRIVER_CLASSES[v] for k, v in FIRE_SUBTYPES.items()}
attr = D[~D.driver.isin(NON_DRIVER)].copy()
EVa = EV[~EV.driver.isin(NON_DRIVER)].copy()
for df in (attr, EVa):
    df["with_fire"] = df.driver.isin(FIRE_LABELS)
    df["driver_group"] = df.driver.replace(PARENT)
GROUPS = [g for g in DRIVER_GROUP_COLORS if g in set(attr.driver_group)]
GCOL = [DRIVER_GROUP_COLORS[g] for g in GROUPS]

PROVISIONAL_FROM = (pd.Timestamp(LATEST_ALERT_DATE) - pd.DateOffset(months=3)).to_period("M")
FULL_YEARS = [y for y in YEARS if pd.Period(f"{y}-12", "M") < PROVISIONAL_FROM]
print(f"{D.area_ha.sum():,.0f} ha PFD in {PF.area_ha.sum():,.0f} ha of peat forest · {len(EV):,} events "
      f"(mean {EV.area_ha.mean():.2f} ha) · {attr.area_ha.sum() / D.area_ha.sum() * 100:.1f} % of PFD area has a driver label")
print("Complete years:", FULL_YEARS, "| provisional from", PROVISIONAL_FROM)

## 13 · Maps
Maps are drawn with matplotlib from arrays fetched with `ee.data.computePixels`, so they render inside the notebook and are saved as PNGs in `outputs_peat_disturbance/figures/`. The display resolution adapts to the AOI size. Large extents are aggregated (sum or max), so small hotspots stay visible.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

FIG_DIR = LOCAL_OUT / "figures"; FIG_DIR.mkdir(exist_ok=True)
def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}_{RUN_TAG}.png", dpi=200, bbox_inches="tight")

EXTENT = (xmin, xmax, ymin, ymax)                 # from §4 (whole degrees around the AOI)

def auto_res(n_bands, max_mb=40):
    area = (xmax - xmin) * (ymax - ymin)
    return max(MAP_RES_DEG, math.sqrt(area * n_bands * 4 / (max_mb * 1e6)))

def fetch(img, bands, res_deg, reducer=None):
    """Fetch bands of an EE image as numpy arrays on a regular lat/lon grid over EXTENT."""
    img = ee.Image(img).select(bands)
    if reducer is not None:
        img = img.reduceResolution(reducer, maxPixels=4096).reproject(
            crs="EPSG:4326", crsTransform=[res_deg, 0, xmin, 0, -res_deg, ymax])
    w, h = int(math.ceil((xmax - xmin) / res_deg)), int(math.ceil((ymax - ymin) / res_deg))
    arr = ee.data.computePixels({
        "expression": img.unmask(-9999).toFloat(), "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": w, "height": h}, "crsCode": "EPSG:4326",
                 "affineTransform": {"scaleX": res_deg, "shearX": 0, "translateX": xmin,
                                     "shearY": 0, "scaleY": -res_deg, "translateY": ymax}}})
    return {b: np.where(arr[b] == -9999, np.nan, arr[b]) for b in bands}

def fc_lines(fc, max_err=2000, limit=5000):
    gj = ee.FeatureCollection(fc).limit(limit).map(lambda f: ee.Feature(f.geometry().simplify(max_err))).getInfo()
    lines = []
    def walk(g):
        if g is None: return
        if g["type"] == "Polygon": lines.append(np.array(g["coordinates"][0]))
        elif g["type"] == "MultiPolygon": lines.extend(np.array(p[0]) for p in g["coordinates"])
        elif g["type"] == "GeometryCollection":
            for sub in g["geometries"]: walk(sub)
    for f in gj["features"]: walk(f["geometry"])
    return lines

COUNTRY_LINES = fc_lines(countries_fc)
PA_LINES = ({"national": fc_lines(pa_nat.filterBounds(aoi), 500), "world": fc_lines(pa_world.filterBounds(aoi), 500)}
            if SHOW_PA_ON_MAPS else {})
PA_STYLE = {"national": dict(color="#d7301f", lw=0.6), "world": dict(color="#6a51a3", lw=0.6, ls="--")}

def basemap(ax, pa=False, title=None):
    for l in COUNTRY_LINES: ax.plot(l[:, 0], l[:, 1], color="black", lw=0.6)
    if pa:
        for k, ls in PA_LINES.items():
            for l in ls: ax.plot(l[:, 0], l[:, 1], **PA_STYLE[k])
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_aspect("equal")
    ax.tick_params(labelsize=7); ax.set_facecolor("#f7f7f7")
    if title: ax.set_title(title, fontsize=10)

def pa_legend():
    return [Line2D([], [], **PA_STYLE["national"], label="National PA"),
            Line2D([], [], **PA_STYLE["world"], label="World PA (WDPA)")] if PA_LINES else []

grid = ee.Image(GRID_ASSET)
print("Map resolution (deg):", round(auto_res(2), 4))

### 13.1 Study area — peat forest extent and peat-map agreement

In [ ]:
res = auto_res(2)
pf = fetch(grid.select("pf_ha"), ["pf_ha"], res, ee.Reducer.mean())["pf_ha"] / 100   # fraction of each cell
agree_img = (peat_src.eq(3).add(1).updateMask(pf_nocountry.And(peat_src.gt(0)))
             .updateMask(country_img.mask()).rename("agree"))                         # 1 non-agreement · 2 agreement
agree = fetch(agree_img, ["agree"], res)["agree"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
im = axes[0].imshow(np.where(pf > 0, pf * 100, np.nan), extent=EXTENT, cmap="Greens", vmin=0, vmax=100)
basemap(axes[0], pa=True, title="Peat forest (2018 forest × combined peat map), % of cell")
fig.colorbar(im, ax=axes[0], shrink=0.6, label="% peat forest")
axes[0].legend(handles=pa_legend(), loc="lower left", fontsize=7)
axes[1].imshow(agree, extent=EXTENT, cmap=ListedColormap(["#a1d99b", "#006d2c"]), vmin=0.5, vmax=2.5,
               interpolation="nearest")
basemap(axes[1], title="Peat map agreement (nearest-neighbour preview)")
axes[1].legend(handles=[Patch(color="#006d2c", label="Agreement (Gumbricht ∩ Crezee)"),
                        Patch(color="#a1d99b", label="Non-agreement (one map only)")], loc="lower left", fontsize=7)
plt.tight_layout(); save(fig, "map_study_area"); plt.show()

### 13.2 Spatial intensity — share of peat forest disturbed per 1 km cell

In [ ]:
lo, hi = INTENSITY_THRESHOLDS_PCT
pf_ha, dist_all = grid.select("pf_ha"), grid.select("dist_ha_all")
ratio = dist_all.divide(pf_ha).multiply(100).updateMask(pf_ha.gt(0)).updateMask(dist_all.gt(0)).rename("ratio")
int_cls = ratio.gte(lo).add(ratio.gt(hi)).add(1).rename("cls")           # 1 low · 2 medium · 3 high

# exact class statistics at 1 km
r = (dist_all.addBands(ee.Image(1).rename("cells")).addBands(int_cls).updateMask(ratio.mask())
     .reduceRegion(ee.Reducer.sum().repeat(2).group(2, "cls"), aoi_bounds, scale=1000, maxPixels=1e10).getInfo())
CLS_NAMES = {1: f"low (<{lo} %)", 2: f"medium ({lo}–{hi} %)", 3: f"high (>{hi} %)"}
INTENSITY = pd.DataFrame([{"intensity": CLS_NAMES[g["cls"]], "PFD (ha)": g["sum"][0], "1 km cells": g["sum"][1]}
                          for g in r["groups"]]).set_index("intensity")
INTENSITY["% of cells"] = pct(INTENSITY["1 km cells"]); INTENSITY["% of PFD"] = pct(INTENSITY["PFD (ha)"])
display(INTENSITY.round(1))

cls = fetch(int_cls, ["cls"], auto_res(1), ee.Reducer.max())["cls"]   # max keeps hotspots visible when aggregated
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(np.where(pf > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#d9f0d3"]), interpolation="nearest")
ax.imshow(cls, extent=EXTENT, cmap=ListedColormap(["#fdae61", "#d7191c", "#2c007a"]), vmin=0.5, vmax=3.5,
          interpolation="nearest")
basemap(ax, pa=True, title=f"Disturbance intensity, {START_YEAR}–{END_YEAR}")
ax.legend(handles=[Patch(color="#d9f0d3", label="Peat forest, undisturbed")] +
          [Patch(color=c, label=CLS_NAMES[i + 1]) for i, c in enumerate(["#fdae61", "#d7191c", "#2c007a"])] +
          pa_legend(), loc="lower left", fontsize=7)
plt.tight_layout(); save(fig, "map_intensity"); plt.show()

h = np.array(ratio.reduceRegion(ee.Reducer.fixedHistogram(0, 100, 100), aoi_bounds, scale=1000,
                                maxPixels=1e10).getInfo()["ratio"])
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(h[:, 0], h[:, 1], width=1, align="edge", color="teal")
for t in INTENSITY_THRESHOLDS_PCT: ax.axvline(t, color="red", ls="--", lw=0.8)
ax.set_xlabel("Disturbance intensity (% of peat forest per 1 km cell)"); ax.set_ylabel("Cells")
ax.set_title("Distribution of cell intensities"); plt.tight_layout(); save(fig, "hist_intensity"); plt.show()

### 13.3 Disturbance by year

In [ ]:
year_bands = [f"dist_ha_{y}" for y in YEARS]
res_y = auto_res(len(year_bands) + 1)
yr = fetch(grid.select(year_bands + ["pf_ha"]), year_bands + ["pf_ha"], res_y, ee.Reducer.sum())
ncol = min(3, len(YEARS)); nrow = math.ceil(len(YEARS) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), squeeze=False)
for ax, y, b in zip(axes.flat, YEARS, year_bands):
    share = np.where(yr["pf_ha"] > 0, yr[b] / yr["pf_ha"] * 100, np.nan)
    ax.imshow(np.where(yr["pf_ha"] > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
    im = ax.imshow(np.where(share > 0, share, np.nan), extent=EXTENT, cmap="inferno_r", vmin=0, vmax=hi / 2)
    basemap(ax, title=f"{y}" + (" (partial)" if y not in FULL_YEARS else ""))
for ax in axes.flat[len(YEARS):]: ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.5, label="% of peat forest disturbed in the year")
fig.suptitle("Annual peat forest disturbance", y=1.0); save(fig, "map_annual"); plt.show()

### 13.4 Driver maps
* **Dominant driver**: the driver with the largest disturbed area in each `DOM_CELL_DEG` cell.
* **Event maps**: one panel per driver, with events sized by size class.

In [ ]:
nx, ny = int(round((xmax - xmin) / DOM_CELL_DEG)), int(round((ymax - ymin) / DOM_CELL_DEG))
cells = EVa.assign(ix=((EVa.lon - xmin) / DOM_CELL_DEG).astype(int).clip(0, nx - 1),
                   iy=((ymax - EVa.lat) / DOM_CELL_DEG).astype(int).clip(0, ny - 1))
by = cells.groupby(["iy", "ix", "driver_group"]).area_ha.sum().reset_index()
tot = by.groupby(["iy", "ix"]).area_ha.sum()
dom = by.sort_values("area_ha").drop_duplicates(["iy", "ix"], keep="last")
dom = dom[dom.set_index(["iy", "ix"]).index.map(tot) >= MIN_CELL_HA]
dom_arr = np.full((ny, nx), np.nan)
dom_arr[dom.iy.to_numpy(), dom.ix.to_numpy()] = dom.driver_group.map({g: i for i, g in enumerate(GROUPS)}).to_numpy()

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(np.where(pf > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
ax.imshow(dom_arr, extent=EXTENT, cmap=ListedColormap(GCOL), vmin=-0.5, vmax=len(GROUPS) - 0.5,
          interpolation="nearest")
basemap(ax, pa=True, title=f"Dominant driver of peat forest disturbance per {DOM_CELL_DEG}° cell, {START_YEAR}–{END_YEAR}")
ax.legend(handles=[Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)] + pa_legend(), loc="lower left", fontsize=7)
plt.tight_layout(); save(fig, "map_dominant_driver"); plt.show()

SIZE_PT = dict(zip(SIZE_LABELS, [2, 9, 30]))
order = EVa.groupby("driver_group").area_ha.sum().reindex(GROUPS).sort_values(ascending=False).index
ncol = 3; nrow = math.ceil(len(order) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), squeeze=False)
for ax, g in zip(axes.flat, order):
    e = EVa[EVa.driver_group == g]
    e = e.sample(min(len(e), MAX_POINTS), random_state=0)
    ax.imshow(np.where(pf > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
    ax.scatter(e.lon, e.lat, s=e.size_class.map(SIZE_PT).astype(float), c=DRIVER_GROUP_COLORS[g],
               alpha=0.6, linewidths=0.2, edgecolors="black")
    basemap(ax, title=f"{g}\n{len(EVa[EVa.driver_group == g]):,} events · {EVa[EVa.driver_group == g].area_ha.sum():,.0f} ha")
for ax in axes.flat[len(order):]: ax.axis("off")
fig.legend(handles=[Line2D([], [], marker="o", ls="", color="grey", markersize=math.sqrt(v), label=k)
                    for k, v in SIZE_PT.items()], loc="lower center", ncol=3, fontsize=8)
fig.suptitle("Disturbance events by driver and size", y=1.0); plt.tight_layout(); save(fig, "map_events_by_driver"); plt.show()

### 13.5 Interactive map (optional, needs `geemap`)

In [ ]:
try:
    import geemap
    m = geemap.Map(center=[(ymin + ymax) / 2, (xmin + xmax) / 2], zoom=6)
    m.addLayer(peat_forest, {"palette": ["1b7837"]}, "Peat forest 2018")
    m.addLayer(int_cls, {"min": 1, "max": 3, "palette": ["fdae61", "d7191c", "2c007a"]}, "Intensity (1 km)")
    m.addLayer(dist_mask, {"palette": ["e31a1c"]}, "PFD pixels", False)
    m.addLayer(driver_idx.updateMask(dist_mask).updateMask(driver_idx.gt(0).And(driver_idx.lt(IDX_MISMATCH))),
               {"min": 1, "max": len(CLASS_CODES), "palette": [DRIVER_GROUP_COLORS.get(PARENT.get(DRIVER_CLASSES[c],
               DRIVER_CLASSES[c]), "#999999").lstrip("#") for c in CLASS_CODES]}, "Driver (10 m)")
    m.addLayer(pa_world.style(color="6a51a3", fillColor="00000000"), {}, "World PAs")
    m.addLayer(pa_nat.style(color="d7301f", fillColor="00000000"), {}, "National PAs")
    display(m)
except ImportError:
    print("geemap not installed — static maps above cover the same layers")

## 14 · Graphs and tables
### 14.1 Extent by country and year

In [ ]:
by_cy = D.pivot_table(index="country", columns="year", values="area_ha", aggfunc="sum", fill_value=0)
pf_area = PF.groupby("country").area_ha.sum()
summary = pd.DataFrame({"PFD (ha)": by_cy.sum(axis=1), "peat forest 2018 (ha)": pf_area})
summary["share of PFD (%)"] = pct(summary["PFD (ha)"])
summary["PFD as % of peat forest"] = summary["PFD (ha)"] / summary["peat forest 2018 (ha)"] * 100
display(by_cy.round(0)); display(summary.sort_values("PFD (ha)", ascending=False).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_cy.T.plot.bar(stacked=True, ax=axes[0], colormap="tab10")
axes[0].set_ylabel("PFD (ha)"); axes[0].set_title("Annual peat forest disturbance by country")
for i, y in enumerate(by_cy.columns):
    if y not in FULL_YEARS: axes[0].annotate("partial", (i, by_cy[y].sum()), ha="center", va="bottom", fontsize=7)
(D.pivot_table(index="year", columns="agreement", values="area_ha", aggfunc="sum", fill_value=0)
   .drop(columns="not peat", errors="ignore").plot.bar(stacked=True, ax=axes[1], color=["#006d2c", "#a1d99b"]))
axes[1].set_ylabel("PFD (ha)"); axes[1].set_title("By peat map agreement")
plt.tight_layout(); save(fig, "bar_extent"); plt.show()

if HAS_VEG:
    display(pct(D[D.veg != "other/unknown"].groupby("veg").area_ha.sum()).round(1).rename("% of palm + hardwood PFD"))

### 14.2 Hotspot districts

In [ ]:
adm = grid.reduceRegions(collection=gaul2, reducer=ee.Reducer.sum(), scale=1000, tileScale=4)
ADM = fc_to_df(adm)
year_cols = [f"dist_ha_{y}" for y in YEARS]
ADM = ADM[["ADM0_NAME", "ADM1_NAME", "ADM2_NAME", "pf_ha", "dist_ha_all"] + year_cols]
ADM["PFD % of peat forest"] = ADM.dist_ha_all / ADM.pf_ha.replace(0, np.nan) * 100
ADM = ADM.sort_values("dist_ha_all", ascending=False)
ADM.to_csv(LOCAL_OUT / f"pfd_by_admin2_{RUN_TAG}.csv", index=False)

top = ADM.head(15).set_index(ADM.head(15).ADM2_NAME + " (" + ADM.head(15).ADM0_NAME.str.replace(
    "Democratic Republic of the Congo", "DRC") + ")")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
top[year_cols].rename(columns=lambda c: c[-4:]).iloc[::-1].plot.barh(stacked=True, ax=axes[0], colormap="viridis")
axes[0].set_xlabel("PFD (ha)"); axes[0].set_title("Top 15 districts by disturbed peat forest")
top["PFD % of peat forest"].iloc[::-1].plot.barh(ax=axes[1], color="#d7191c")
axes[1].set_xlabel("% of district peat forest disturbed"); axes[1].set_title("Relative to peat forest area")
plt.tight_layout(); save(fig, "bar_districts"); plt.show()

### 14.3 Seasonality and temporal hotspots

In [ ]:
mt = (D.groupby(["year", "month"]).area_ha.sum().unstack("month").reindex(columns=range(1, 13), fill_value=0).fillna(0))
mpct = mt.div(mt.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for y in mpct.index:
    ls = "-" if y in FULL_YEARS else "--"
    axes[0].plot(range(1, 13), mpct.loc[y], marker="o", ls=ls, label=str(y))
    axes[1].plot(range(1, 13), mt.loc[y], marker="o", ls=ls, label=str(y))
axes[0].axhline(TEMPORAL_HOTSPOT_PCT, color="grey", lw=0.8, ls=":")
axes[0].set_ylabel("% of the year's PFD"); axes[1].set_ylabel("PFD (ha)")
for a in axes: a.set_xticks(range(1, 13)); a.set_xlabel("Month")
axes[0].legend(ncol=3, fontsize=8)
fig.suptitle("Monthly peat forest disturbance (dashed = partial / provisional year)")
plt.tight_layout(); save(fig, "line_monthly"); plt.show()

fig, ax = plt.subplots(figsize=(9, 2.8))
im = ax.imshow(mpct.to_numpy(), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(12)); ax.set_xticklabels(range(1, 13)); ax.set_yticks(range(len(mpct))); ax.set_yticklabels(mpct.index)
for (i, j), v in np.ndenumerate(mpct.to_numpy()):
    if v > TEMPORAL_HOTSPOT_PCT: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, label="% of year"); ax.set_title(f"Temporal hotspots (months > {TEMPORAL_HOTSPOT_PCT} % labelled)")
plt.tight_layout(); save(fig, "heat_monthly"); plt.show()

temporal = pd.DataFrame({"Jan–Jun share (%)": mpct[list(range(1, 7))].sum(axis=1),
                         f"hotspot months (>{TEMPORAL_HOTSPOT_PCT} %)":
                             pd.Series({y: [m for m in range(1, 13) if mpct.loc[y, m] > TEMPORAL_HOTSPOT_PCT] for y in mpct.index})})
display(temporal.round(1))

### 14.4 Protection status
Shares are shown both by event count (events intersected with PA and concession polygons) and by pixel area.

In [ ]:
def protection_table(df, weight):
    g = (lambda by: df.groupby(by)[weight].sum()) if weight else (lambda by: df.groupby(by).size())
    pa = pct(g("pa"))
    out = {"in any PA": 100 - pa.get("none", 0),
           "in world PA": pa.get("world", 0) + pa.get("national+world", 0),
           "in national PA": pa.get("national", 0) + pa.get("national+world", 0)}
    if HAS_CONC:
        out["inside concessions"] = pct(g("conc")).get("inside", 0)
    return pd.Series(out)

prot = pd.DataFrame({"% of events": protection_table(EV, None), "% of area": protection_table(D, "area_ha")})
display(prot.round(1))

pa_country = D.pivot_table(index="country", columns="pa", values="area_ha", aggfunc="sum", fill_value=0)
pa_country = pa_country.div(pa_country.sum(axis=1), axis=0) * 100
pa_drv = attr.pivot_table(index="driver_group", columns="pa", values="area_ha", aggfunc="sum", fill_value=0)
pa_drv = pa_drv.div(pa_drv.sum(axis=1), axis=0).reindex(GROUPS) * 100
cols = [c for c in ["none", "national", "world", "national+world"] if c in pa_country]
pal = {"none": "#d9d9d9", "national": "#d7301f", "world": "#6a51a3", "national+world": "#fc8d59"}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pa_country[cols].plot.barh(stacked=True, ax=axes[0], color=[pal[c] for c in cols])
pa_drv[[c for c in cols if c in pa_drv]].plot.barh(stacked=True, ax=axes[1], color=[pal[c] for c in cols if c in pa_drv], legend=False)
axes[0].set_title("PFD by protection status, per country"); axes[1].set_title("…per driver")
for a in axes: a.set_xlabel("% of PFD area")
plt.tight_layout(); save(fig, "bar_protection"); plt.show()

if HAS_CONC:
    display(pd.crosstab(attr.driver_group, attr.conc, values=attr.area_ha, aggfunc="sum", normalize="index")
            .mul(100).reindex(GROUPS).round(1))

### 14.5 Proximity to peat edges, roads and rivers

In [ ]:
prox_edge = cum_share(EV, "edge_bin", EDGE_BINS_KM, None)
prox_acc = pd.DataFrame({"road or river": cum_share(EV, "access_bin", ACCESS_BINS_KM, None),
                         "road": cum_share(EV, "road_bin", ACCESS_BINS_KM, None),
                         "river": cum_share(EV, "river_bin", ACCESS_BINS_KM, None)})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(EDGE_BINS_KM, prox_edge.iloc[:-1], marker="o", color="#e34a33", label="Peat edges")
for col, c in zip(prox_acc.columns, ["#7f0000", "#1b9e77", "#2b4bd8"]):
    axes[0].plot(ACCESS_BINS_KM, prox_acc[col].iloc[:-1], marker="o", color=c, label=f"{col.capitalize()} networks")
axes[0].set_xlabel("Distance (km)"); axes[0].set_ylabel("Cumulative % of disturbance events"); axes[0].set_ylim(0, 101)
axes[0].legend(); axes[0].set_title("All disturbances")
for g, c in zip(GROUPS, GCOL):
    e = EVa[EVa.driver_group == g]
    if len(e) >= 20:
        axes[1].plot(EDGE_BINS_KM, cum_share(e, "edge_bin", EDGE_BINS_KM, None).iloc[:-1], marker=".", color=c, label=g)
axes[1].set_xlabel("Distance to peat edge (km)"); axes[1].set_ylabel("Cumulative % of events"); axes[1].set_ylim(0, 101)
axes[1].legend(fontsize=7); axes[1].set_title("By driver")
plt.tight_layout(); save(fig, "line_proximity"); plt.show()

display(pd.DataFrame({"peat edge": prox_edge}).T.round(1)); display(prox_acc.T.round(1))
display(EVa.groupby("driver_group").apply(lambda g: pd.Series({
    "≤1 km of peat edge": cum_share(g, "edge_bin", EDGE_BINS_KM, None)["≤1 km"],
    "≤5 km of road/river": cum_share(g, "access_bin", ACCESS_BINS_KM, None)["≤5 km"],
    "≤5 km of road": cum_share(g, "road_bin", ACCESS_BINS_KM, None)["≤5 km"],
    "≤5 km of river": cum_share(g, "river_bin", ACCESS_BINS_KM, None)["≤5 km"]})).reindex(GROUPS).round(1))

### 14.6 Drivers — composition, trends, seasonality and fire

In [ ]:
status = D.assign(status=np.where(D.driver.isin(NON_DRIVER), D.driver, "labelled"))
print("Driver attribution status by year (% of PFD area):")
display(status.pivot_table(index="year", columns="status", values="area_ha", aggfunc="sum", fill_value=0)
        .pipe(lambda t: t.div(t.sum(axis=1), axis=0) * 100).round(1))

def driver_table(ev):
    t = ev.groupby(["country", "driver_group"]).agg(events=("area_ha", "size"), area_ha=("area_ha", "sum"))
    t["area % within country"] = t.area_ha / t.groupby(level=0).area_ha.transform("sum") * 100
    t["area % of total"] = t.area_ha / t.area_ha.sum() * 100
    return t.unstack(0)
print("Drivers by country (events):"); display(driver_table(EVa).round(1))

dc = attr.pivot_table(index="country", columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=GROUPS, fill_value=0)
dy = attr.pivot_table(index="year", columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=GROUPS, fill_value=0)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
dc.div(dc.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, ax=axes[0], color=GCOL, legend=False)
axes[0].set_xlabel("% of PFD area"); axes[0].set_title("Driver composition by country")
dy.plot.bar(stacked=True, ax=axes[1], color=GCOL, legend=False)
axes[1].set_ylabel("PFD (ha)"); axes[1].set_title("Annual PFD by driver")
dy.div(dy.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[2], color=GCOL)
axes[2].set_ylabel("% of PFD area"); axes[2].set_title("Driver shares by year")
axes[2].legend(bbox_to_anchor=(1, 1), fontsize=7)
plt.tight_layout(); save(fig, "bar_drivers"); plt.show()

# monthly % of each driver relative to the year's total of all drivers
annual = attr.groupby("year").area_ha.sum()
dm = attr.groupby(["driver_group", "year", "month"]).area_ha.sum().div(annual, level="year") * 100
present = [g for g in GROUPS if g in dm.index.get_level_values(0)]
ncol = 3; nrow = math.ceil(len(present) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3 * nrow), squeeze=False, sharex=True)
for ax, g in zip(axes.flat, present):
    t = dm.loc[g].unstack("year").reindex(range(1, 13)).fillna(0)
    for y in t.columns:
        ax.plot(t.index, t[y], marker=".", ls="-" if y in FULL_YEARS else "--", label=str(y))
    ax.set_title(g, fontsize=9, color=DRIVER_GROUP_COLORS[g]); ax.set_ylabel("% of year")
for ax in axes.flat[len(present):]: ax.axis("off")
axes.flat[0].legend(fontsize=7, ncol=2); plt.xticks(range(1, 13))
fig.suptitle("Monthly distribution by driver (scales differ)"); plt.tight_layout(); save(fig, "line_monthly_by_driver"); plt.show()

ag = attr[attr.driver_group.isin(PARENT.values())]
if len(ag):
    fy = (ag[ag.with_fire].pivot_table(index="year", columns="driver_group", values="area_ha", aggfunc="sum")
          / ag.pivot_table(index="year", columns="driver_group", values="area_ha", aggfunc="sum") * 100)
    fm = (ag[ag.with_fire].groupby("month").area_ha.sum() / ag.groupby("month").area_ha.sum() * 100).reindex(range(1, 13))
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
    fy.plot(marker="o", ax=axes[0], color=[DRIVER_GROUP_COLORS[c] for c in fy.columns])
    axes[0].set_ylabel("% with fire"); axes[0].set_title("Agricultural clearing with fire, by year")
    fm.plot.bar(ax=axes[1], color="#d95f02"); axes[1].set_ylabel("% with fire"); axes[1].set_title("…by month")
    plt.tight_layout(); save(fig, "fire_subtype"); plt.show()

### 14.7 Disturbance events — size, peat map agreement and forest type

In [ ]:
print(f"{len(EV):,} events · {EV.area_ha.sum():,.0f} ha · mean {EV.area_ha.mean():.2f} ha · "
      f"{EV.below_mmu.mean() * 100:.1f} % below the {MMU_HA} ha MMU")

size_sum = EV.groupby("size_class", observed=False).agg(events=("area_ha", "size"), area_ha=("area_ha", "sum"))
size_sum["% of events"] = pct(size_sum.events); size_sum["% of area"] = pct(size_sum.area_ha)
display(size_sum.round(1))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(EV.area_ha.clip(upper=10), bins=np.arange(0, 10.05, 0.1), color="teal")
axes[0].axvline(EV.area_ha.mean(), color="blue", ls="--", label=f"mean {EV.area_ha.mean():.2f} ha")
axes[0].set_xlabel("Event area (ha, clipped at 10)"); axes[0].set_ylabel("Events"); axes[0].legend()
axes[0].set_title("Event size distribution")
sd = EVa.pivot_table(index="size_class", columns="driver_group", values="area_ha", aggfunc="sum", observed=False,
                     fill_value=0).reindex(columns=GROUPS, fill_value=0)
sd.div(sd.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], color=GCOL, legend=False, rot=0)
axes[1].set_ylabel("% of area"); axes[1].set_title("Driver shares by event size")
ad = EVa[EVa.agreement != "not peat"].pivot_table(index="agreement", columns="driver_group", values="area_ha",
                                                  aggfunc="sum", fill_value=0).reindex(columns=GROUPS, fill_value=0)
ad.div(ad.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[2], color=GCOL, rot=0)
axes[2].set_ylabel("% of area"); axes[2].set_title("Driver shares by peat map agreement")
axes[2].legend(bbox_to_anchor=(1, 1), fontsize=7)
plt.tight_layout(); save(fig, "bar_events"); plt.show()

if HAS_VEG:
    vd = EVa[EVa.veg != "other/unknown"].pivot_table(index="veg", columns="driver_group", values="area_ha",
                                                     aggfunc="sum", fill_value=0).reindex(columns=GROUPS, fill_value=0)
    ax = vd.div(vd.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, color=GCOL, figsize=(9, 2.5))
    ax.set_xlabel("% of area"); ax.set_title("Driver shares by peat forest type"); ax.legend(bbox_to_anchor=(1, 1), fontsize=7)
    plt.tight_layout(); save(ax.figure, "bar_vegetation"); plt.show()

## 15 · Optional — stratified sample to check the WUR driver labels
This draws a sample per driver for visual review in Collect Earth Online or QGIS. For each event, look at Planet NICFI and Sentinel-2 monthly imagery for up to two years after the alert. For each class, look for:
* **Small-scale agriculture:** regrowth followed by harvest, or a pattern matching adjacent fields.
* **Logging:** canopy or bare patch with no regrowth.
* **Road:** a lasting linear opening.
* **Flooding:** along a river with no fields nearby.

In [ ]:
N_PER_DRIVER = 50
sample = (EVa[~EVa.below_mmu].groupby("driver", group_keys=False)
          .apply(lambda g: g.sample(min(len(g), N_PER_DRIVER), random_state=42)))
sample = sample.assign(alert_date=pd.to_datetime(sample.year.astype(str) +
                       (sample.t_first % 1000).astype(int).astype(str).str.zfill(3), format="%Y%j"))
sample = sample.assign(review_until=sample.alert_date + pd.DateOffset(years=2))
sample = sample[["lon", "lat", "country", "driver", "alert_date", "review_until", "area_ha", "size_class",
                 "agreement", "pa"]].reset_index(drop=True)
sample.insert(0, "plotid", sample.index + 1)
sample.to_csv(LOCAL_OUT / f"driver_validation_sample_{RUN_TAG}.csv", index=False)
print(sample.driver.value_counts())

## 16 · Notes and limitations
* **Recent alerts.** About the last three months before `LATEST_ALERT_DATE` contain unconfirmed alerts, and the current year is partial. Plots dash these years.
* **Attaching WUR labels to RADD alerts.** WUR alerts integrate RADD, GLAD-L and GLAD-S2 (plus DIST from 2026), so the WUR date is often the earliest detection by any system. The mosaic keeps the most recent WUR alert per pixel. A RADD alert is labelled only if the WUR date falls within `DRIVER_DATE_TOLERANCE_DAYS` of the RADD date, otherwise it counts as `date_mismatch`. The status table in §14.6 shows how much area this affects.
* **`wur` disturbance mode.** Integrated alerts pick up more small and early disturbances than RADD alone, so totals are higher.
* **Forest baseline.** RADD's 2018 primary humid forest baseline excludes secondary and savanna-mosaic forest, for example in southern DRC and the Batéké plateaux.
* **Peat maps outside the Cuvette.** The Crezee map covers the central Congo Basin, so for peat elsewhere in ROC and DRC (e.g. coastal Kouilou or eastern DRC) the mask depends on Gumbricht (231 m). `agreement` in every table lets you restrict to areas where both maps agree.
* **MMU artefacts.** Intersecting 0.1 ha alerts with the peat and forest masks creates sub-MMU fragments. `below_mmu` flags them, and the validation sample skips them.
* **Distances** are Euclidean distance transforms at 50 m (edges) and 100 m (roads/rivers). Bin boundaries are accurate to about one pixel.
* **Events** are 8-connected pixels over the whole period. Events crossing a 1° processing tile are split.
* **Roads.** GRIP4 misses most forest tracks. Kleinschroth et al. (2019) or national road layers give more realistic accessibility.
* **Maps** are display products. Fetched arrays are aggregated to fit the request size, and the agreement panel is a nearest-neighbour preview. All statistics come from the 10 m and 1 km reductions, not the map arrays.